# Day 27 Tutorial：论文到数据字典

## Goal

把来源描述拆成样本定义、字段角色、可见性和合并检查。

## Setup

使用人工来源描述，不联网、不下载数据、不训练模型。

In [1]:
import pandas as pd

source_table = pd.DataFrame([
    {
        'source_id': 'source_A',
        'task_type': 'regression',
        'sample_definition': 'one measured object',
        'target_definition': 'continuous target',
        'unit': 'pending',
        'raw_data_obtained': False,
        'license_status': 'pending',
    }
])
source_table

,source_id,task_type,sample_definition,target_definition,unit,raw_data_obtained,license_status
0,source_A,regression,one measured object,continuous target,pending,False,pending


## Steps

### 1. 每个字段一行，显式记录角色和 query 前可见性

In [2]:
field_dictionary = pd.DataFrame([
    ('sample_id', 'identifier', 'string', None, True),
    ('feature_1', 'input', 'float', 'pending', True),
    ('target_value', 'target', 'float', 'pending', False),
    ('batch_id', 'group', 'string', None, True),
], columns=['field', 'role', 'dtype', 'unit', 'visible_during_query'])
field_dictionary

,field,role,dtype,unit,visible_during_query
0,sample_id,identifier,string,None,True
1,feature_1,input,float,pending,True
2,target_value,target,float,pending,False
3,batch_id,group,string,None,True


### 2. 让合并结论来自检查项

In [3]:
merge_checks = {
    'same_sample_definition': True,
    'same_target_meaning': True,
    'unit_compatible': False,
    'protocol_compatible': False,
    'license_checked': False,
}
may_merge = all(merge_checks.values())
pending = [name for name, passed in merge_checks.items() if not passed]
print('may_merge:', may_merge)
print('pending:', pending)

may_merge: False
pending: ['unit_compatible', 'protocol_compatible', 'license_checked']


## Checks

核对标签隐藏、未知信息和合并默认值。

In [4]:
target_row = field_dictionary.query("role == 'target'").iloc[0]
assert target_row['visible_during_query'] == False
assert source_table.loc[0, 'unit'] == 'pending'
assert not source_table.loc[0, 'raw_data_obtained']
assert may_merge is False
assert set(pending) == {'unit_compatible', 'protocol_compatible', 'license_checked'}
print('Checks passed: schema only; no training rows.')

Checks passed: schema only; no training rows.


## Next Steps

关闭 Notebook，独立完成练习；遇到未知信息时回到来源核实，不填造默认值。